# Chapter 13 — Markov Chains: The Future Forgets the Past

Companion notebook for the [probintro textbook](https://josephausterweil.github.io/probintro/intro2/13_markov_chains/).

Chibany chooses tonkatsu or hamburger each day with a one-step habit. We build the chain as a **transition matrix**, **sample** sequences from it, and find its **stationary distribution** by power iteration and as the eigenvalue-1 eigenvector.

## Setup

In [ ]:
!pip install genjax -q
print('✅ ready')

## Power iteration: two starts converge to 70/30

Most Markov-chain machinery is plain linear algebra: multiply a distribution by the transition matrix `P` over and over.

In [ ]:
import jax.numpy as jnp

# Chibany's transition matrix. Rows = today (T, H); columns = tomorrow (T, H).
P = jnp.array([[0.65, 0.35],
               [0.82, 0.18]])

def power_iterate(v, P, steps):
    """Multiply the distribution v by P, `steps` times, recording every iterate."""
    rows = [v]
    for _ in range(steps):
        v = v @ P
        rows.append(v)
    return jnp.stack(rows)

traj_T = power_iterate(jnp.array([1.0, 0.0]), P, 20)   # always start on tonkatsu
traj_H = power_iterate(jnp.array([0.0, 1.0]), P, 20)   # always start on hamburger

for k in [0, 1, 2, 5, 20]:
    print(f"step {k:2d}: from T ({traj_T[k,0]:.3f}, {traj_T[k,1]:.3f})   "
          f"from H ({traj_H[k,0]:.3f}, {traj_H[k,1]:.3f})")

## π is the eigenvalue-1 eigenvector

Power iteration sneaks up on an eigenvector; we can also ask for it directly and confirm they agree. Left eigenvectors of `P` are eigenvectors of `P.T`.

In [ ]:
import numpy as np

vals, vecs = np.linalg.eig(np.array(P).T)   # left eigenvectors of P = eigvecs of P^T
idx = int(np.argmin(np.abs(vals - 1.0)))    # the eigenvalue equal to 1
pi = np.real(vecs[:, idx])
pi = pi / pi.sum()                          # normalize so it sums to 1

print(f"eigenvalue   = {np.real(vals[idx]):.3f}")
print(f"pi (eigen)   = ({pi[0]:.3f}, {pi[1]:.3f})")
print(f"pi (run it)  = ({traj_T[20,0]:.3f}, {traj_T[20,1]:.3f})")

## The matrix is a sampler (GenJAX)

A transition matrix doesn't only *describe* probabilities — it *draws samples*. `categorical` picks a state from **log-probabilities**, so we pass `jnp.log(P)[state]`. We fix the step count as a plain Python int inside a factory so JAX can unroll the loop.

In [ ]:
import jax
import jax.random as jr
from genjax import gen, categorical

LOGP = jnp.log(P)   # categorical wants log-probabilities; row `state` = current state

def make_chain(n_steps):
    @gen
    def chain(start):
        state = start
        states = [state]
        for t in range(n_steps):
            state = categorical(LOGP[state]) @ f"x_{t}"
            states.append(state)
        return jnp.array(states)
    return chain

chain20 = make_chain(20)

labels = ["T", "H"]   # plain Python list — JAX arrays can't hold strings
keys = jr.split(jr.key(0), 5)
runs = jax.vmap(lambda k: chain20.simulate(k, (0,)).get_retval())(keys)
for r in runs:
    print(" ".join(labels[int(s)] for s in r))

## The long-run 70/30 by sampling

For a long run we carry the state through a `jax.lax.scan` instead of unrolling thousands of addresses, then count the tonkatsu days.

In [ ]:
def run_long(key, start, n):
    def step(state, k):
        nxt = jr.categorical(k, LOGP[state])
        return nxt, nxt
    _, visited = jax.lax.scan(step, start, jr.split(key, n))
    return visited

visited = run_long(jr.key(1), 0, 5000)
frac_tonkatsu = float(jnp.mean((visited == 0).astype(float)))
print(f"long-run fraction tonkatsu (5000 steps) ~ {frac_tonkatsu:.2f}")

## A three-state chain — same method

The procedure doesn't care how many states there are. State 2 ends up the loneliest, exactly as the arrows predict.

In [ ]:
A = jnp.array([[0.0, 0.1, 0.9],
               [0.5, 0.0, 0.5],
               [0.8, 0.2, 0.0]])

from_1 = power_iterate(jnp.array([1.0, 0.0, 0.0]), A, 40)
from_2 = power_iterate(jnp.array([0.0, 1.0, 0.0]), A, 40)
print("from state 1:", tuple(round(float(x), 2) for x in from_1[40]))
print("from state 2:", tuple(round(float(x), 2) for x in from_2[40]))

## Try it yourself

1. Change Chibany's loyalty: set `P[0]` to `[0.50, 0.50]` and re-run power iteration. Does tonkatsu still dominate?
2. Confirm `A`'s stationary distribution with `np.linalg.eig(np.array(A).T)` — find the eigenvalue-1 eigenvector and normalize it.